In [11]:
# Minimal, NumPy-only toy for the CAN-sandwich Procrustes setup.
# Goal: given canonical angles for A, B, T, build the block variables C_a,S_a,... and
# show how to form the Procrustes subproblem for R (and analogously for R_tilde).
#
# This is intentionally simple: no JAX, no Qiskit—just matrices you can poke at.

import numpy as np


def cos_sin_diags(angles):
    """Return diagonal matrices C=diag(cos a_i), S=diag(sin a_i)."""
    a = np.array(angles, dtype=float)
    C = np.diag(np.cos(a))
    S = np.diag(np.sin(a))
    return C, S


def residual_L(Ca, Sa, Cb, Sb, Ct, St, R, Rtil):
    """Frobenius residual for the two coupled equations:
    Ct == Ca R Cb - Sa Rtil Sb
    St == Ca R Sb + Sa Rtil Cb
    """
    term1 = Ct - (Ca @ R @ Cb - Sa @ Rtil @ Sb)
    term2 = St - (Ca @ R @ Sb + Sa @ Rtil @ Cb)
    return np.linalg.norm(term1, "fro") ** 2 + np.linalg.norm(term2, "fro") ** 2


def procrustes_SO3(Y, X):
    """Solve min_R || R X - Y ||_F with R in SO(3)."""
    U, _, Vt = np.linalg.svd(Y @ X.T)
    R = U @ np.diag([1, 1, np.linalg.det(U @ Vt)]) @ Vt
    return R


def step_solve_R(Ca, Sa, Cb, Sb, Ct, St, Rtil):
    # Stack columns so that RX ≈ Y
    X = np.hstack([Cb, Sb])  # 3x6
    Y = np.linalg.pinv(Ca) @ np.hstack(
        [Ct + Sa @ Rtil @ Sb, St - Sa @ Rtil @ Cb]
    )  # 3x6
    return procrustes_SO3(Y, X)


def step_solve_Rtil(Ca, Sa, Cb, Sb, Ct, St, R):
    # Stack columns so that Rtil Xtil ≈ Ytil
    Xtil = np.hstack([-Sb, Cb])  # 3x6
    Ytil = np.linalg.pinv(Sa) @ np.hstack([Ct - Ca @ R @ Cb, St - Ca @ R @ Sb])  # 3x6
    return procrustes_SO3(Ytil, Xtil)


# ---- Toy numeric inputs (angles in radians) ----
# Feel free to change these; they just need to be small-ish to avoid extreme ill-conditioning.
a = (0.25, 0.0, 1.0)  # A: (a1, a2, a3)
b = (0.25, 0.0, 1.0)  # B: (b1, b2, b3)
t = (
    0.0,
    0.0,
    1.0,
)  # T: (t1, t2, t3) <-- arbitrary; may not be exactly feasible for one step

Ca, Sa = cos_sin_diags(a)
Cb, Sb = cos_sin_diags(b)
Ct, St = cos_sin_diags(t)

# Start with identity seed for Rtil
Rtil0 = np.eye(3)
R0 = step_solve_R(Ca, Sa, Cb, Sb, Ct, St, Rtil0)

# One alternating update
Rtil1 = step_solve_Rtil(Ca, Sa, Cb, Sb, Ct, St, R0)
R1 = step_solve_R(Ca, Sa, Cb, Sb, Ct, St, Rtil1)

res0 = residual_L(Ca, Sa, Cb, Sb, Ct, St, R0, Rtil0)
res1 = residual_L(Ca, Sa, Cb, Sb, Ct, St, R1, Rtil1)

print("Inputs (angles):")
print("  A =", a)
print("  B =", b)
print("  T =", t)
print("\nDiagonal blocks:")
print("  C_a=\n", Ca, "\n  S_a=\n", Sa)
print("  C_b=\n", Cb, "\n  S_b=\n", Sb)
print("  C_t=\n", Ct, "\n  S_t=\n", St)

print("\nFirst Procrustes solve (R | Rtil=I):")
print("  R0=\n", R0)
print("  residual L(R0, I) =", res0)

print("\nAfter one alternating step:")
print("  Rtil1=\n", Rtil1)
print("  R1=\n", R1)
print("  residual L(R1, Rtil1) =", res1)


Inputs (angles):
  A = (0.25, 0.0, 1.0)
  B = (0.25, 0.0, 1.0)
  T = (0.0, 0.0, 1.0)

Diagonal blocks:
  C_a=
 [[0.96891242 0.         0.        ]
 [0.         1.         0.        ]
 [0.         0.         0.54030231]] 
  S_a=
 [[0.24740396 0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.84147098]]
  C_b=
 [[0.96891242 0.         0.        ]
 [0.         1.         0.        ]
 [0.         0.         0.54030231]] 
  S_b=
 [[0.24740396 0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.84147098]]
  C_t=
 [[1.         0.         0.        ]
 [0.         1.         0.        ]
 [0.         0.         0.54030231]] 
  S_t=
 [[0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.84147098]]

First Procrustes solve (R | Rtil=I):
  R0=
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
  residual L(R0, I) = 1.164230264482975

After one alternating step:
  Rtil1=
 [[-1.  0.  0.]
 [ 0. -1.  0.]
 

In [ ]:
from gulps.core.invariants import GateInvariants
from qiskit.circuit.library import CXGate, iSwapGate

GateInvariants.from_unitary(iSwapGate().power(1 / 2)).makhlin

[0.25000000000000017, 0.0, 1.0000000000000004]

In [6]:
R = np.eye(3)
Rtil = np.eye(3)
for k in range(50):
    R_prev, Rtil_prev = R.copy(), Rtil.copy()
    R = step_solve_R(Ca, Sa, Cb, Sb, Ct, St, Rtil)
    Rtil = step_solve_Rtil(Ca, Sa, Cb, Sb, Ct, St, R)
    L = residual_L(Ca, Sa, Cb, Sb, Ct, St, R, Rtil)
    if np.linalg.norm(R - R_prev) + np.linalg.norm(Rtil - Rtil_prev) < 1e-12:
        break
print("final residual:", L)


final residual: 0.24483487621925457
